In this part of the rush, you need to prepare everything that is used in the classes and methods above in a Jupyter Notebook (recipes.ipynb).

### Imports

In [1]:
import joblib
import requests
import itertools
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm
from urllib.parse import quote_plus
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import mean_squared_error, accuracy_score, precision_score, roc_auc_score, recall_score
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, StackingClassifier, BaggingClassifier, VotingClassifier

### Preprocessing

In [2]:
df = pd.read_csv('../data/epi_r.csv')

X = df.drop(columns=['title', 'rating', 'calories', 'protein', 'fat', 'sodium', '#cakeweek', '#wasteless', '22-minute meals', '3-ingredient recipes', '30 days of groceries', 'advance prep required', 'alabama', 'alaska', 'alcoholic', 'leftovers', 'california', 'dominican republic', 'low cholesterol', 'pan-fry', 'fruit', 'stew'], axis=1)
y = df['rating']

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21)
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.2, random_state=21)

### Regression

"
Try different algorithms and their hyperparameters for rating prediction. Choose the best on cross-validation and find the score (RMSE) on the test subsample.
Try different ensembles and their hyperparameters. Choose the best on cross-validation and find the score on the test subsample.
Calculate the RMSE for a naive regressor that predicts the average rating.
"

In [23]:
linreg = LinearRegression()
linreg_params = [{'fit_intercept': [True, False]
}]
gs_linreg = GridSearchCV(estimator=linreg, param_grid=linreg_params, scoring='neg_mean_squared_error', cv=2, n_jobs=7)

tree = DecisionTreeRegressor()
tree_params = [{'max_depth': [10, 20, 30],
                'min_samples_split': [2, 5, 8, 10],
                'min_samples_leaf': [1, 2, 4],
                'random_state': [21]
    }]
gs_tree = GridSearchCV(estimator=tree, param_grid=tree_params, scoring='neg_mean_squared_error', cv=2, n_jobs=7)

rf = RandomForestRegressor(n_jobs = -1)
rf_params = {
    'n_estimators': [50, 100, 150],
    'max_depth': [10, 20, 30],
    'min_samples_split': [2, 5, 8],
    'random_state': [21]  
}
gs_rf = GridSearchCV(estimator=rf, param_grid=rf_params, scoring='neg_mean_squared_error', cv=2, n_jobs=7, verbose=1)

grids = [gs_linreg, gs_tree, gs_rf]
grid_dict = {
    0: "Linear Regression",
    1: "Decision Tree", 
    2: "Random Forest"
}

In [24]:
class ModelSelection:
    def __init__(self, grids, grid_dict):
        self.grids = grids
        self.grid_dict = grid_dict
        self.model_params = []

    def choose(self, X_train, y_train, X_valid, y_valid, mode):
        for i, gs in enumerate((self.grids)):
            model = self.grid_dict[i]
            print(f"Estimator: {model}")

            gs.fit(X_train, y_train)
            best_model = gs.best_estimator_
            best_params = gs.best_params_
            y_pred_train = best_model.predict(X_train)
            y_pred_valid = best_model.predict(X_valid)
            
            if mode == "Regression":
                rmse_train = np.sqrt(mean_squared_error(y_pred_train, y_train))
                rmse_valid = np.sqrt(mean_squared_error(y_pred_valid, y_valid))
                print(f"Best params: {best_params}\nTraining rmse: {rmse_train:.3f}\nValidation set rmse for best params: {rmse_valid:.3f}\n")
                self.model_params.append({"model": model, 
                                        'params': best_params, 
                                        'valid_score': rmse_valid})
            elif mode == "Classifier":
                accuracy_train = best_model.score(X_train, y_train)
                accuracy_valid = best_model.score(X_valid, y_valid)
                print(f"Best params: {best_params}\nBest training accuracy: {accuracy_train:.3f}\nValidation set accuracy score for best params: {accuracy_valid:.3f}\n")

                self.model_params.append({"model": model, 
                                        'params': best_params, 
                                        'valid_score': accuracy_valid})
             
    def best_results(self):
        best_models = pd.DataFrame(self.model_params)

        return best_models

In [25]:
class Finalize:
    def __init__(self, model):
        self.model = model
    def final_score(self, X_train, y_train, X_test, y_test, mode):
        self.model.fit(X_train, y_train)
        if mode == "Regression":
            y_pred = self.model.predict(X_test)
            rmse = np.sqrt(mean_squared_error(y_pred, y_test))
            print(f"RMSE of the final model is {rmse}")
        elif mode == "Classifier":
            final_accuracy = self.model.score(X_test, y_test)
            print(f"Accuracy of the final model is {final_accuracy}")
    def save_model(self, path):
        dump(self.model, path, compress=9)
        print(f"{self.model} was successfully saved!")

In [26]:
model_regression = ModelSelection(grids, grid_dict)

In [28]:
model_regression.choose(X_train, y_train, X_valid, y_valid, mode="Regression")

Estimator: Linear Regression


Best params: {'fit_intercept': False}
Training rmse: 1.350
Validation set rmse for best params: 34676175573.328

Estimator: Decision Tree
Best params: {'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 10, 'random_state': 21}
Training rmse: 1.183
Validation set rmse for best params: 1.241

Estimator: Random Forest
Fitting 2 folds for each of 27 candidates, totalling 54 fits


[Parallel(n_jobs=7)]: Using backend LokyBackend with 7 concurrent workers.
[Parallel(n_jobs=7)]: Done  36 tasks      | elapsed:  1.3min
[Parallel(n_jobs=7)]: Done  54 out of  54 | elapsed:  2.1min finished


Best params: {'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 150, 'random_state': 21}
Training rmse: 0.962
Validation set rmse for best params: 1.212



In [ ]:
best_results = model_regression.best_results().sort_values("valid_score")
best_results

,model,params,valid_score
2,Random Forest,"{'max_depth': 20, 'min_samples_split': 2, 'n_e...",1.211854e+00
1,Decision Tree,"{'max_depth': 10, 'min_samples_leaf': 4, 'min_...",1.240769e+00
0,Linear Regression,{'fit_intercept': True},5.103725e+07


In [ ]:
best_model_regression = RandomForestRegressor(max_depth=20, min_samples_split=2, n_estimators=150, random_state=21)
final_rmse = Finalize(model=best_model_regression)

In [ ]:
final_rmse.final_score(X_train, y_train, X_test, y_test, mode="Regression")

RMSE of the final model is 1.2076686434677448


In [ ]:
df_copy = df.copy()
average_rating = np.mean(df["rating"])
df_copy["naive_predict"] = average_rating 
naive_rmse = np.sqrt(mean_squared_error(df_copy["naive_predict"], df["rating"]))

naive_rmse

1.3407952575187467

### Classification

Binarize the target column by rounding the ratings to the closest integer. This will be your classes.

In [4]:
y = df['rating'].round(0)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.2, random_state=21, stratify=y_train)

In [ ]:
svm = SVC()
svm_params =  [{'kernel': ['linear', 'rbf', 'sigmoid'], 
                'C':[0.01, 0.1, 1, 10, 100], 
                'gamma': ['scale', 'auto'], 
                'random_state':[21], 
                'probability':[True]}]
gs_svm = GridSearchCV(estimator=svm, param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=-1)

tree = DecisionTreeClassifier()
tree_params = [{'max_depth': range(5, 21, 3),
                'min_samples_split': range(5, 16, 2),
                'random_state': [21],
                'criterion': ["entropy", "gini"],
}]
gs_tree = GridSearchCV(estimator=tree, param_grid=tree_params, scoring='accuracy', cv=2, n_jobs=-1)

rf = RandomForestClassifier()
rf_params = {
    'criterion': ["entropy", "gini"],
    'n_estimators': range(50, 251, 50),
    'max_depth': range(25, 36, 2),
    'min_samples_split': range(5, 10),
    'random_state': [21]
}
gs_rf = GridSearchCV(estimator=rf, param_grid=rf_params, scoring='accuracy', cv=2, n_jobs=-1)

grids = [gs_svm, gs_tree, gs_rf]
grid_dict = {
    0: "SVM",
    1: "Decision Tree", 
    2: "Random Forest"
}

Try different algorithms and their hyperparameters for class prediction. Choose the best on cross-validation and find the score (accuracy) on the test subsample.
Compare the metrics using accuracy. Calculate the accuracy of a naive classificator that predicts the most common class.

In [ ]:
model_classifier = ModelSelection(grids, grid_dict)

In [ ]:
model_classifier.choose(X_train, y_train, X_valid, y_valid, mode="Classifier")

Estimator: SVM
Best params: {'C': 100, 'gamma': 'auto', 'kernel': 'sigmoid', 'probability': True, 'random_state': 21}
Best training accuracy: 0.680
Validation set accuracy score for best params: 0.677

Estimator: Decision Tree
Best params: {'criterion': 'gini', 'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 8, 'random_state': 21}
Best training accuracy: 0.697
Validation set accuracy score for best params: 0.672

Estimator: Random Forest
Best params: {'criterion': 'entropy', 'max_depth': 30, 'min_samples_split': 2, 'n_estimators': 150, 'random_state': 21}
Best training accuracy: 0.839
Validation set accuracy score for best params: 0.692




In [ ]:
best_models_classfier = (model_classifier.best_results()).sort_values('valid_score', ascending=False)
best_models_classfier

,model,params,valid_score
2,Random Forest,"{'criterion': 'entropy', 'max_depth': 30, 'min...",0.692116
0,SVM,"{'C': 100, 'gamma': 'auto', 'kernel': 'sigmoid...",0.676846
1,Decision Tree,"{'criterion': 'gini', 'max_depth': 10, 'min_sa...",0.671860


Choose the best on cross-validation and find the score (accuracy) on the test subsample

In [ ]:
best_model = RandomForestClassifier(criterion='entropy', max_depth=30, min_samples_split=2, n_estimators=150, random_state=21)
final_classification = Finalize(model=best_model)

In [ ]:
final_classification.final_score(X_train, y_train, X_test, y_test, mode="Classifier")

Accuracy of the final model is 0.6898529045125904


Compare the metrics using accuracy. Calculate the accuracy of a naive classificator that predicts the most common class.

In [ ]:
classifier = df.copy()
naive_predict = pd.Series([y.mode()[0]]* y.shape[0])
naive_accuracy = accuracy_score(naive_predict, y)

naive_accuracy 

0.6576900059844405

What is worse: to predict a bad rating which is good in real life, or to predict a good rating which is bad in real life? Replace accuracy with the appropriate metric.

In [ ]:
y_pred = best_model.predict(X_test)
precision = precision_score(y_pred, y_test, average='weighted')
recall = recall_score(y_pred, y_test, average='weighted')

y_proba = best_model.predict_proba(X_test)
roc_auc = roc_auc_score(y_test,y_proba, multi_class='ovr', average='weighted')

print(f"precision is {precision:.5f}\nrecall is {recall:.5f}\nroc_auc is {roc_auc:.5f}")

precision is 0.94699
recall is 0.68985
roc_auc is 0.75145


Try different algorithms and their hyperparameters for class prediction with the new metric. Choose the best and find the score on the test subsample.

In [6]:
bins = [0, 1, 3, 5]
labels = ["bad", "so-so", "great"]
y_train = pd.cut(y_train, bins=bins, labels=labels, include_lowest=True)
y_valid = pd.cut(y_valid, bins=bins, labels=labels, include_lowest=True)
y_test = pd.cut(y_test, bins=bins, labels=labels, include_lowest=True)


In [ ]:
model_classifier.choose(X_train, y_train, X_valid, y_valid, mode="Classifier")

Estimator: SVM
Best params: {'C': 1, 'gamma': 'scale', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.820
Validation set accuracy score for best params: 0.807

Estimator: Decision Tree
Best params: {'criterion': 'gini', 'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 2, 'random_state': 21}
Best training accuracy: 0.819
Validation set accuracy score for best params: 0.801

Estimator: Random Forest
Best params: {'criterion': 'entropy', 'max_depth': 30, 'min_samples_split': 2, 'n_estimators': 100, 'random_state': 21}
Best training accuracy: 0.867
Validation set accuracy score for best params: 0.811




In [ ]:
best_model = RandomForestClassifier(criterion='entropy', max_depth=30, min_samples_split=2, n_estimators=100, random_state=21)
final_classification = Finalize(model=best_model)

In [ ]:
final_classification.final_score(X_train, y_train, X_test, y_test, mode="Classifier")

Accuracy of the final model is 0.8080279232111692


Try different ensembles and their hyperparameters. Choose the best and find the score on the test subsample.

In [ ]:
#VotingClassifier

svm = SVC(C=10, class_weight=None, gamma='auto', kernel='rbf', random_state=21, probability=True)
tree = DecisionTreeClassifier(min_samples_split=2, min_samples_leaf=2, random_state=21, criterion='gini', max_depth=10)
forest = RandomForestClassifier(criterion='entropy', max_depth=30, min_samples_split=2, n_estimators=100, random_state=21, n_jobs=-1)

eclf = VotingClassifier(estimators=[('tree', tree), ('forest', forest)], voting='hard', weights=[1,2])
eclf.fit(X_train, y_train)

y_pred = eclf.predict(X_valid)
accuracy = eclf.score(X_valid, y_valid)
precision = precision_score(y_valid, y_pred, average='weighted')
recall = recall_score(y_valid, y_pred, average='weighted')

print(f"accuracy is {accuracy:.5f}\nprecision is {precision:.5f}\nrecall is {recall:.5f}")

accuracy is 0.81053
precision is 0.80772
recall is 0.81053


In [ ]:
#BaggingClassifier

model = RandomForestClassifier(criterion='entropy', max_depth=30, min_samples_split=2, n_estimators=100, random_state=21, n_jobs=-1)
bagging_model = BaggingClassifier(base_estimator=model, n_estimators=50, random_state=21, bootstrap=False, max_samples=0.85)
bagging_model.fit(X_train, y_train)

y_pred = bagging_model.predict(X_valid)
accuracy = bagging_model.score(X_valid, y_valid)
precision = precision_score(y_valid, y_pred, average='weighted')

print(accuracy, precision)

0.8086631349330009 0.8240454435292573


In [ ]:
#StackingClassifier

results = []
for n_splits in [2, 5, 8]:
    for passthrough in [True, False]:
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=21)
        estimators = [('svm', LinearSVC(C=1, penalty='l2', random_state=21)),
        ('tree', DecisionTreeClassifier(min_samples_split=2, min_samples_leaf=2, random_state=21, criterion='gini', max_depth=10))]

        clf = StackingClassifier(estimators=estimators, final_estimator=RandomForestClassifier(criterion='entropy', max_depth=30, min_samples_split=2, n_estimators=100, random_state=21, n_jobs=-1), passthrough=passthrough, cv=skf)
        clf.fit(X_train, y_train)

        y_pred = clf.predict(X_valid)
        accuracy = clf.score(X_valid, y_valid)
        precision = precision_score(y_valid, y_pred, average='weighted')
        recall = recall_score(y_valid, y_pred, average='weighted')

        metrics = {"n_splits": n_splits,
                   'passthrough': passthrough,
                   'accuracy': accuracy,
                   'precision': precision,
                   'recall': recall}
        
        results.append(metrics)
        

results = pd.DataFrame(results)

In [ ]:
results

,n_splits,passthrough,accuracy,precision,recall
0,2,True,0.809910,0.822315,0.809910
1,2,False,0.805235,0.708626,0.805235
2,5,True,0.811779,0.824000,0.811779
3,5,False,0.798691,0.709489,0.798691
4,8,True,0.815519,0.830680,0.815519
5,8,False,0.798380,0.714381,0.798380


In [ ]:
skf = StratifiedKFold(n_splits=8, shuffle=True, random_state=21)
estimators = [('svm', LinearSVC(C=1, penalty='l2', random_state=21)),
        ('tree', DecisionTreeClassifier(min_samples_split=2, min_samples_leaf=2, random_state=21, criterion='gini', max_depth=10))]
best_ensemle = StackingClassifier(estimators=estimators, final_estimator=RandomForestClassifier(criterion='entropy', max_depth=30, min_samples_split=2, n_estimators=100, random_state=21, n_jobs=-1), passthrough=True, cv=skf)
best_ensemle.fit(X_train, y_train)

y_pred = best_ensemle.predict(X_test)
accuracy = best_ensemle.score(X_test, y_test)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

print(f"accuracy is {accuracy:.5f}\nprecision is {precision:.5f}\nrecall is {recall:.5f}")


accuracy is 0.80878
precision is 0.80639
recall is 0.80878


### Decision

Decide what is better to use: the regression model or the classification. Save the best model. You will use it in the program.

Decision: classification is better than regression

In [ ]:
final_model = Finalize(model=best_ensemle)
final_model.save_model(path='../data/best_model.joblib')

StackingClassifier(cv=StratifiedKFold(n_splits=8, random_state=21, shuffle=True),
                   estimators=[('svm', LinearSVC(C=1, random_state=21)),
                               ('tree',
                                DecisionTreeClassifier(max_depth=10,
                                                       min_samples_leaf=2,
                                                       random_state=21))],
                   final_estimator=RandomForestClassifier(criterion='entropy',
                                                          max_depth=30,
                                                          n_jobs=-1,
                                                          random_state=21),
                   passthrough=True) was successfully saved!


## Nutrition Facts

Collect all the nutrition facts for the ingredients from your prepared and filtered dataset (only ingredient columns) into a dataframe. Use [the following API](https://fdc.nal.usda.gov/api-guide.html) for that. 

In [ ]:
ingredients = X.columns.tolist()
API_KEY = 'TWyKaEA5P0DEoCy2JhRY4azqU1WgknG7owXGA2ua'
BASE_URL = "https://api.nal.usda.gov/fdc/v1"

nutrient_data_by_ingredient = {}

for ingredient in ingredients:
    search_url = f"{BASE_URL}/foods/search?api_key={API_KEY}&query={ingredient}"
    try:
        search_response = requests.get(search_url)
        search_response.raise_for_status()
        search_results = search_response.json()

        fdc_id = None
        if search_results and 'foods' in search_results and len(search_results['foods']) > 0:
            fdc_id = search_results['foods'][0]['fdcId']

        if fdc_id:
            food_detail_url = f"{BASE_URL}/food/{fdc_id}?api_key={API_KEY}"
            detail_response = requests.get(food_detail_url)
            detail_response.raise_for_status()
            food_data = detail_response.json()

            data = []
            if 'foodNutrients' in food_data:
                for nutrient in food_data['foodNutrients']:
                    nutrient_name = nutrient.get('nutrient', {}).get('name')
                    unit_name = nutrient.get('nutrient', {}).get('unitName')
                    amount = nutrient.get('amount')
                    if nutrient_name and amount is not None:
                        data.append({"name": nutrient_name, "unitName": unit_name, "amount": amount})
            
            nutrient_data_by_ingredient[ingredient] = data

    except requests.exceptions.HTTPError as e:
        print(f"Ошибка HTTP для {ingredient}: {e}")
    except requests.exceptions.RequestException as e:
        print(f"Ошибка запроса для {ingredient}: {e}")
    except json.JSONDecodeError:
        print(f"Ошибка декодирования JSON для {ingredient}.")
    except Exception as e:
        print(f"Неожиданная ошибка для {ingredient}: {e}")

In [ ]:
flattened_records = []
for ingredient_name, nutrient_list in nutrient_data_by_ingredient.items():
    for nutrient_info in nutrient_list:
        nutrient_name = nutrient_info['name']

        flattened_records.append({
            'Ingredient': ingredient_name,
            'Nutrient': nutrient_name,
            'Amount': nutrient_info['amount'],
            'Unit': nutrient_info['unitName']
        })

df_flat = pd.DataFrame(flattened_records)

def convert_amount_to_mcg(row):
    amount = row['Amount']
    unit = row['Unit']
    
    if pd.isna(amount):
        return 0

    unit_lower = str(unit).lower()

    if unit_lower in ('mcg', 'µg'):
        return amount
    elif unit_lower == 'mg':
        return amount * 1000
    elif unit_lower == 'g':
        return amount * 1_000_000
    elif unit_lower == 'iu':
        return amount * 0.025
    else:
        return 0

df_flat['Amount_mcg'] = df_flat.apply(convert_amount_to_mcg, axis=1)

In [ ]:
df_pivot = df_flat.pivot_table(index='Nutrient', columns='Ingredient', values='Amount_mcg')

df_final = df_pivot.fillna(0)

# df_final.to_csv('../data/api_nutrients.csv')

In [ ]:
nutrients_data = pd.read_csv('../data/api_nutrients.csv')
nutrients_data.set_index('Nutrient', inplace=True)

Transform all the values into % of the daily value. Keep only nutrients that
exist in [this](https://drive.google.com/file/d/1jn0t5tU_RgOpq4wcO-uS4D0_NAP6MwHz/view?usp=sharing) and [that](https://drive.google.com/file/d/1bmdZGB0QwND2BD3XlC1JswL7AdnTJHLT/view?usp=sharing) table. 

In [ ]:
daily_values = {
    "Vitamin A, RAE": 900.0,
    "Vitamin C, total ascorbic acid": 90000.0,
    "Calcium, Ca": 1300000.0,
    "Iron, Fe": 18000.0,
    "Vitamin D (D2 + D3), International Units": 20.0,
    "Vitamin E (alpha-tocopherol)": 15000.0,
    "Vitamin K (phylloquinone)": 120.0,
    "Thiamin": 1200.0,
    "Riboflavin": 1300.0,
    "Niacin": 16000.0,
    "Vitamin B-6": 1700.0,
    "Folate, total": 400.0,
    "Vitamin B-12": 2.4,
    "Biotin": 30.0,
    "Pantothenic acid": 5000.0,
    "Phosphorus, P": 1250000.0,
    "Iodine, I": 150.0,
    "Magnesium, Mg": 420000.0,
    "Zinc, Zn": 11000.0,
    "Selenium, Se": 55.0,
    "Copper, Cu": 900.0,
    "Manganese, Mn": 2300.0,
    "Chromium, Cr": 35.0,
    "Molybdenum": 45.0,
    "Chloride, Cl": 2300000.0,
    "Potassium, K": 4700000.0,
    "Choline, total": 550000.0,
    "Protein": 50000000.0,
    "Total lipid (fat)": 78000000.0,
    "Fatty acids, total saturated": 20000000.0,
    "Cholesterol": 300000.0,
    "Carbohydrate, by difference": 275000000.0,
    "Sodium, Na": 2300000.0,
    "Fiber, total dietary": 28000000.0,
    "Sugars, added": 50000000.0
}

In [ ]:
nutrients_to_keep = list(daily_values.keys())
df_filtered = nutrients_data.loc[nutrients_data.index.intersection(nutrients_to_keep)]

daily_values_series = pd.Series(daily_values)

df_percentage = (df_filtered.divide(daily_values_series, axis=0) * 100).fillna(0)
df_percentage = df_percentage.map(lambda x: f"{x:.2f}%" if pd.notna(x) else "0.00%")

Save the transformed dataframe into a CSV file that you will use in your main program

In [ ]:
df_percentage.to_csv('../data/percentages.csv')

## Similar Recipes

* For each recipe from the dataset, collect the URL from epicurious.com with its details (if there is no URL for that recipe, skip it).
* Save the new dataframe to a CSV file that you will use in your main program.

In [ ]:
base_search_url = "https://www.epicurious.com/search?q="
recipe_links = {}

unique_titles = df['title'].unique()

for recipe_title in unique_titles:
    recipe_title = recipe_title.strip()
    recipe_link = None
    
    encoded_title = quote_plus(recipe_title)
    search_url = f"{base_search_url}{encoded_title}"
    
    print(f"Поиск: '{recipe_title}' по URL: {search_url}")
    
    try:
        response = requests.get(search_url)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.text, 'html.parser')
        
        first_recipe = soup.find('div', {'data-testid': "ClampWrapper"})
        first_recipe = first_recipe.find('a')
        recipe_name = first_recipe.find('h2').get_text()
        
        if first_recipe and recipe_name == recipe_title.strip():
            recipe_link = "https://www.epicurious.com" + first_recipe.get('href')
            print(f"Найдена ссылка для '{recipe_title}': {recipe_link}")
        else:
            print(f"Ссылка на рецепт не найдена для '{recipe_title}'.")
            
        recipe_links[recipe_title] = recipe_link
        
    except requests.exceptions.RequestException as e:
        print(f"Ошибка запроса для '{recipe_title}': {e}")
    except Exception as e:
        print(f"Произошла ошибка при парсинге для '{recipe_title}': {e}")


In [ ]:
links_df = pd.read_csv('../data/only_links.csv').dropna().set_index('title')

df = pd.concat([df['title'], X], axis=1)
df = pd.concat([df, y], axis=1)
df['title'] = df['title'].str.strip()
df.set_index('title', inplace=True)

df = df.join(links_df)

In [ ]:
df.to_csv('../data/recipes.csv')

### Bonus part

For each combination of breakfast, lunch and dinner from dataset check whether most nutritional needs covered without overtaking.

In [34]:
recipes = pd.read_csv('../data/recipes.csv', index_col=0)

In [29]:
percentages = pd.read_csv('../data/percentages.csv', index_col=0)
for col in percentages.columns:
    percentages[col] = percentages[col].str.replace('%', '').astype(float) / 100

In [ ]:
common_ingredients = list(
    set(recipes.columns).intersection(percentages.columns)
)
recipes_spc = recipes[common_ingredients]
percentages = percentages[common_ingredients]

df_total = recipes_spc.dot(percentages.T)
df_total.to_csv('../data/nutrients_per_meal.csv')

In [32]:
df_total = pd.read_csv('../data/nutrients_per_meal.csv')

In [ ]:
from joblib import Parallel, delayed
PARTIAL_RESULTS_FILE = "../data/valid_combinations.joblib"
loaded_valid_combinations = []

breakfast_recipes = recipes[recipes['breakfast'] == 1].index.to_list()
lunch_recipes = recipes[recipes['lunch'] == 1].index.to_list()
dinner_recipes = recipes[recipes['dinner'] == 1].index.to_list()
total_combinations = len(breakfast_recipes) * len(lunch_recipes) * len(dinner_recipes)
total_nutrients = df_total.shape[1]
min_nutrients_in_range = total_nutrients / 1.66
t_min = 0.7
t_max = 1
eps = 10**-6
min_total_rating = 10


def check_and_return_combination(b_recipe_name, l_recipe_name, d_recipe_name, df_nutrients, min_nutrients, target_min, target_max, eps, min_total_rating):
    breakfast_nutrients_np = df_nutrients.loc[b_recipe_name].values
    lunch_nutrients_np = df_nutrients.loc[l_recipe_name].values
    dinner_nutrients_np = df_nutrients.loc[d_recipe_name].values
    total_rating = (recipes.loc[b_recipe_name, 'rating'] + 
                    recipes.loc[l_recipe_name, 'rating'] +
                    recipes.loc[d_recipe_name, 'rating'])

    if total_rating <= min_total_rating:
        return None

    combined_nutrients = breakfast_nutrients_np + lunch_nutrients_np + dinner_nutrients_np
    nutrients_in_range_count = int(((combined_nutrients >= target_min) & (combined_nutrients <= target_max) | (combined_nutrients <= eps)).sum())

    if nutrients_in_range_count >= min_nutrients:
        return {
            'Breakfast_recipe': b_recipe_name,
            'Lunch_recipe': l_recipe_name,
            'Dinner_recipe': d_recipe_name,
            'Nutrients': nutrients_in_range_count
        }
    return None

print(f"проверка {total_combinations} комбинаций...")

CHUNK_SIZE = 1000000
current_combinations_processed = 0

all_combinations_generator = itertools.product(breakfast_recipes, lunch_recipes, dinner_recipes)

with tqdm(total=total_combinations, desc="Обработка комбинаций") as pbar:
    while True:
        batch_to_process_now = []
        for _ in range(CHUNK_SIZE):
            try:
                combo = next(all_combinations_generator)
                batch_to_process_now.append(combo)
                pbar.update(1)
            except StopIteration:
                break

        if not batch_to_process_now and current_combinations_processed >= total_combinations:
            break

        batch_results = Parallel(n_jobs=-1)(
            delayed(check_and_return_combination)(b, l, d, df_total, min_nutrients_in_range, t_min, t_max, eps, min_total_rating)
            for b, l, d in batch_to_process_now
        )

        valid_in_batch = [r for r in batch_results if r is not None]

        loaded_valid_combinations.extend(valid_in_batch)

        try:
            print(f"Количество комбинаций для сохранения: {len(loaded_valid_combinations)}")
            joblib.dump(loaded_valid_combinations, PARTIAL_RESULTS_FILE)
        except Exception as e:
            print(f"Ошибка при сохранении частичных результатов: {e}")

        current_combinations_processed += len(batch_to_process_now)